In [ ]:
# ================= 1. 初始化硬件 =================
print("初始化底盘与摄像头...")
bot = LOBOROBOT()
bot.t_stop(0)  # 确保开机静止

# 初始化云台角度
PAN_CH = 10    # 左右通道
TILT_CH = 9    # 上下通道
current_pan = 80
current_tilt = 0
bot.set_servo_angle(PAN_CH, current_pan)
bot.set_servo_angle(TILT_CH, current_tilt)

In [ ]:
# ================= 2. 视频流全局缓存机制 =================
# 全局变量，存放最新的一帧 JPEG 图像，确保多客户端/网络波动时视频不卡顿、无延迟积压
global_frame = None
#线程锁，更新画面 or 上传画面时 上锁，保证画面的完整性
frame_lock = threading.Lock()

# 视频采集子线程
def video_capture_thread():
    global global_frame
    # 优化后的 GStreamer 管道
    gstreamer_pipeline = (
        "libcamerasrc ! "
        "video/x-raw, width=1280, height=720, framerate=20/1 ! "
        "videoconvert ! "
        "video/x-raw, format=BGR ! "  # 强制转换为 OpenCV 原生支持的 BGR 格式，提高效率
        "appsink drop=true max-buffers=1" # 【关键】丢弃旧帧，缓冲区只留1帧，保证画面0延迟
    )
    
    # 告诉 OpenCV 使用 GStreamer 后端来解析这个管道
    cap = cv2.VideoCapture(gstreamer_pipeline, cv2.CAP_GSTREAMER)

    # 检查是否成功打开
    if not cap.isOpened():
        print("❌ 严重错误：无法通过 GStreamer 打开摄像头！请检查管道设置。")
        return

    while True:
        ret, frame = cap.read()
        if ret:
            # 如果原始画面颠倒，翻转画面 (根据你实际摄像头的安装方向决定，-1是中心对称翻转)
            frame = cv2.flip(frame, -1)

            # 将画面编码为 JPEG 
            encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 82]
            ret_enc, buffer = cv2.imencode('.jpg', frame, encode_param)

            if ret_enc:
                with frame_lock:
                    global_frame = buffer.tobytes()
        else:
            time.sleep(0.01)

# 启动视频采集线程
t_cam = threading.Thread(target=video_capture_thread)
t_cam.daemon = True
t_cam.start()


In [ ]:
# ================= 3. HTTP 服务器 =================
#Flask Web框架的标准开头：建立一个名为app的网站服务器程序
app = Flask(__name__)
# 关闭 Flask 默认的访问日志，避免控制台被刷屏
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

def generate_stream():
    """视频流生成器，推送最新的 global_frame"""
    while True:
        with frame_lock:
            jpeg = global_frame

        if jpeg is not None:
            # 拼接 MJPEG 协议数据包
            yield (b'--frame\r\n'
                   b'Content-Type: image/jpeg\r\n\r\n' + jpeg + b'\r\n')
        # 稍微休眠，限制最高推送帧率约30fps，避免耗尽 WiFi 带宽
        time.sleep(0.03)

#若有人访问了网址，则执行video_feed函数
@app.route('/mycamera')
def video_feed():
    # 响应 HTTP 请求，类型设为 multipart/x-mixed-replace
    return Response(generate_stream(), mimetype='multipart/x-mixed-replace; boundary=frame')


In [ ]:
# ================= 4. 获取本机IP并开启网络服务 =================
def get_ip_address():
    try:
        s = socket(AF_INET, SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except error:
        return "127.0.0.1"

ip = get_ip_address()
print(f"✅ 树莓派视频流地址: http://{ip}:8080/mycamera")
print(f"✅ 树莓派指令接收 UDP 端口: {ip}:2001")

# 在子线程中启动 Flask 服务器
def run_flask():
    app.run(host='0.0.0.0', port=8080, threaded=True, use_reloader=False)

t_flask = threading.Thread(target=run_flask)
t_flask.daemon = True
t_flask.start()

# 开启 UDP 指令接收服务器
udp_server = socket(AF_INET, SOCK_DGRAM)
udp_server.bind(('0.0.0.0', 2001))  # 绑定 0.0.0.0 接受任何来源的控制包

In [ ]:
# ================= 5. 主循环：接收指令并执行 =================
print("等待 PC 端发送控制指令...")
speed = 50 # 默认速度

try:
    while True:
        # 接收数据，阻塞等待
        data_recv, addr = udp_server.recvfrom(1024)
        cmd = data_recv.decode('utf-8').strip()
        # print(f"收到指令: {cmd}") # 调试用

        # --- 运动控制 ---
        if cmd == "UP":          bot.t_up(speed, 0)
        elif cmd == "DOWN":      bot.t_down(speed, 0)
        elif cmd == "LEFT_MOVE": bot.moveLeft(speed, 0)
        elif cmd == "RIGHT_MOVE":bot.moveRight(speed, 0)
        elif cmd == "TURN_L":    bot.turnLeft(speed, 0)
        elif cmd == "TURN_R":    bot.turnRight(speed, 0)
        elif cmd == "UP_LEFT":   bot.forward_Left(speed, 0)
        elif cmd == "UP_RIGHT":  bot.forward_Right(speed,0)
        elif cmd == "DOWN_LEFT": bot.backward_Left(speed,0)
        elif cmd == "DOWN_RIGHT":bot.backward_Right(speed,0)
        elif cmd == "STOP":      bot.t_stop(0)

        # --- 舵机(云台)控制 ---
        elif cmd == "CAM_UP":
            current_tilt = max(0, current_tilt - 10)
            bot.set_servo_angle(TILT_CH, current_tilt)
        elif cmd == "CAM_DOWN":
            current_tilt = min(180, current_tilt + 10)
            bot.set_servo_angle(TILT_CH, current_tilt)
        elif cmd == "CAM_LEFT":
            current_pan = min(180, current_pan + 10)
            bot.set_servo_angle(PAN_CH, current_pan)
        elif cmd == "CAM_RIGHT":
            current_pan = max(0, current_pan - 10)
            bot.set_servo_angle(PAN_CH, current_pan)

except KeyboardInterrupt:
    print("程序被手动终止。")

finally:
    bot.t_stop(0) # 停车
    udp_server.close() # 释放网络端口
    print("资源已安全释放。")
    # 注：守护线程(daemon)会随着主程序的退出自动销毁，不需要手动 stop Flask